## <font color='blue'>Fine-Tuning de LLM Open-Source Para App de Assistente de Perguntas e Respostas</font>

In [3]:
# 1. Atualiza Numpy e Scipy para a versão 2.x (Isso resolve o "ValueError: numpy.dtype size changed" no NLTK)
!pip uninstall -y numpy scipy
!pip install -q 'numpy<2.1' 'scipy<1.15'

# 2. Desinstala pacotes não utilizados no fine-tuning de NLP que geram alertas de dependência
!pip uninstall -y torchaudio torchvision textblob ydata-profiling peft

# 3. Faz o downgrade do fsspec para a versão exata que o datasets 2.20.0 exige
!pip install -q fsspec==2024.5.0

# 4. Garante a instalação limpa do restante do ecossistema harmonizado
!pip install -q torch==2.3.1 transformers==4.41.2 tokenizers==0.19.1 datasets==2.20.0 evaluate==0.4.2 rouge_score==0.1.2 sentencepiece==0.2.0 accelerate==0.31.0 nltk==3.8.1

Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: textblob 0.19.0
Uninstalling textblob-0.19.0:
  Successfully uninstalled textblob-0.19.0
Found existing installation: ydata-profiling 4.18.4
Uninstalling ydata-profiling-4.18.4:
  Successfully uninstalled ydata-profiling-4.18.4
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed

## Instalando e Carregando Pacotes

In [4]:
# Imports
import numpy as np
import nltk
import shutil
import evaluate
import transformers
import datasets
from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Remove as pastas, se já existirem. Isso evita erros ao executar o jupyter a partir da segunda vez em diante
try:
    shutil.rmtree('logs_treino')
    shutil.rmtree('resultados_treino')
    shutil.rmtree('modelo_salvo')
except:
    print('As pastas não existem ou já foram removidas!')

As pastas não existem ou já foram removidas!


## Carregando os Dados

In [6]:
# Define o nome do arquivo
nome_arquivo = "/kaggle/input/datasets/danielrrezende/datasetdsa/cap9/dataset.csv"

In [7]:
# Carrega os dados
ft_dataset = load_dataset('csv', data_files = nome_arquivo, delimiter = ',')

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
# Divisão em treino e teste com proporção 80/20
ft_dataset = ft_dataset["train"].train_test_split(test_size = 0.2)

In [9]:
# Visualiza o formato do dataset
ft_dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 2993
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 749
    })
})

## Tokenizador e LLM Open-Source

https://huggingface.co/google/flan-t5-base

In [10]:
# Carrega o tokenizador
tokenizador = T5Tokenizer.from_pretrained("google/flan-t5-small", legacy = False)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [11]:
# Visualiza o tokenizador
tokenizador

T5Tokenizer(name_or_path='google/flan-t5-small', vocab_size=32000, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<extra_id_42>', '<extra_id_43>', 

In [12]:
# Carrega o LLM pré-treinado
modelo = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [13]:
# Visualiza a arquitetura do LLM
modelo

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [14]:
# Data collator para concatenar tokenizador e modelo
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizador, model = modelo)

In [15]:
# Visualiza o data collator
data_collator

DataCollatorForSeq2Seq(tokenizer=T5Tokenizer(name_or_path='google/flan-t5-small', vocab_size=32000, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '

## Pré-Processamento

In [16]:
# Todos os inputs vão receber como prefixo: "answer the question" (responda a questão)
prefix = "answer the question: "

In [17]:
# Função de pré-processamento
def ft_fn_preprocess(data):

    # Concatena o prefixo a cada pergunta na lista de perguntas fornecida em data["question"]
    inputs = [prefix + doc for doc in data["question"]]

    # Usa o tokenizer para converter as perguntas processadas em tokens com um comprimento máximo de 128,
    # truncando as que forem mais longas
    model_inputs = tokenizador(inputs, max_length = 128, truncation = True)

    # Tokeniza as respostas fornecidas em data["answer"] com um comprimento máximo de 512,
    # truncando as que forem mais longas
    labels = tokenizador(text_target = data["answer"], max_length = 512, truncation = True)

    # Adiciona os tokens das respostas como rótulos no dicionário de entradas do modelo
    model_inputs["labels"] = labels["input_ids"]

    # Retorna o dicionário processado contendo tanto as entradas tokenizadas quanto os rótulos
    return model_inputs

In [18]:
# Aplica a função de pré-processamento ao dataset gerando o dataset tokenizado
ft_dataset_tokenized = ft_dataset.map(ft_fn_preprocess, batched = True)

Map:   0%|          | 0/2993 [00:00<?, ? examples/s]

Map:   0%|          | 0/749 [00:00<?, ? examples/s]

In [19]:
# Visualiza o dataset tokenizado
ft_dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2993
    })
    test: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 749
    })
})

In [20]:
ft_dataset_tokenized['train']['question'][0]

'Q: i worked for a restaurant for 5 and sum years. i felt i was forced to quit after being yelled at for something petty. the boss was chef and the wife was cashier. the kids ran as floaters. i never called out always there on time. i did everything there. the daughter for some reason was picking on me. pointing things out to her father to where he was now yelling at me. one day he approached me to tell me not to make it a habit to bring my 15 year old daughter to work. she sat at a table to eat food that she paid for. to me she was a paying customer. how could that have bothered him? Another time i called out due to being sick. i went back the next day and when i asked for my tip money she said "i am sorry but because you called out we are now disciplining everyone who calls out by not giving you the tips for the WEEK" i only got a 20 min break for the 8 and a half hours that i was there. (10;30am-7pm) i was forced to quit. i felt i was being discriminated against. '

In [21]:
ft_dataset_tokenized['train']['answer'][0]

"A:Under California law, your situation may involve several legal considerations. First, if your employer's conduct towards you was based on your race, gender, religion, or another protected characteristic, it could constitute illegal discrimination. Second, California labor laws require employers to provide certain break periods based on the number of hours worked, and failure to comply could be a labor violation. Third, withholding tips as a form of discipline is generally not permissible, as tips are considered the property of the employee who earned them. Fourth, the issue with your daughter being treated unfairly as a paying customer might not directly fall under employment law, but it could contribute to a hostile work environment if it was part of broader discriminatory or harassing behavior. Lastly, if you felt compelled to resign due to intolerable working conditions, this might be construed as constructive dismissal. It's advisable to consult with an employment attorney to di

In [22]:
ft_dataset_tokenized['train']['input_ids'][0]

[1525,
 8,
 822,
 10,
 1593,
 10,
 3,
 23,
 1279,
 21,
 3,
 9,
 2062,
 21,
 305,
 11,
 4505,
 203,
 5,
 3,
 23,
 1800,
 3,
 23,
 47,
 5241,
 12,
 10399,
 227,
 271,
 3,
 63,
 693,
 26,
 44,
 21,
 424,
 3,
 102,
 16142,
 5,
 8,
 7930,
 47,
 6380,
 11,
 8,
 2512,
 47,
 1723,
 972,
 5,
 8,
 1082,
 4037,
 38,
 3,
 12660,
 277,
 5,
 3,
 23,
 470,
 718,
 91,
 373,
 132,
 30,
 97,
 5,
 3,
 23,
 410,
 762,
 132,
 5,
 8,
 3062,
 21,
 128,
 1053,
 47,
 8915,
 30,
 140,
 5,
 3,
 15989,
 378,
 91,
 12,
 160,
 2353,
 12,
 213,
 3,
 88,
 47,
 230,
 3,
 63,
 15,
 12013,
 44,
 140,
 5,
 80,
 239,
 3,
 88,
 15319,
 140,
 12,
 817,
 140,
 59,
 12,
 143,
 34,
 3,
 9,
 7386,
 12,
 830,
 82,
 627,
 215,
 1]

In [23]:
ft_dataset_tokenized['train']['labels'][0]

[71,
 10,
 5110,
 588,
 1826,
 973,
 6,
 39,
 1419,
 164,
 7789,
 633,
 1281,
 4587,
 7,
 5,
 1485,
 6,
 3,
 99,
 39,
 6152,
 31,
 7,
 3498,
 1587,
 25,
 47,
 3,
 390,
 30,
 39,
 1964,
 6,
 7285,
 6,
 5562,
 6,
 42,
 430,
 5046,
 16115,
 6,
 34,
 228,
 11708,
 6016,
 9192,
 257,
 5,
 5212,
 6,
 1826,
 5347,
 3786,
 1457,
 7713,
 12,
 370,
 824,
 1733,
 8811,
 3,
 390,
 30,
 8,
 381,
 13,
 716,
 1279,
 6,
 11,
 3338,
 12,
 9095,
 228,
 36,
 3,
 9,
 5347,
 12374,
 5,
 9879,
 6,
 28,
 21647,
 2316,
 38,
 3,
 9,
 607,
 13,
 7998,
 19,
 2389,
 59,
 7630,
 7,
 2317,
 6,
 38,
 2316,
 33,
 1702,
 8,
 785,
 13,
 8,
 3490,
 113,
 4964,
 135,
 5,
 21670,
 6,
 8,
 962,
 28,
 39,
 3062,
 271,
 4260,
 18216,
 120,
 38,
 3,
 9,
 3788,
 884,
 429,
 59,
 1461,
 1590,
 365,
 4311,
 973,
 6,
 68,
 34,
 228,
 4139,
 12,
 3,
 9,
 24550,
 161,
 1164,
 3,
 99,
 34,
 47,
 294,
 13,
 3,
 13627,
 9192,
 6546,
 42,
 30139,
 53,
 3889,
 5,
 3,
 19807,
 6,
 3,
 99,
 25,
 1800,
 3,
 287,
 14528,
 12,
 3,
 60,
 6732

## Definindo a Métrica de Avaliação do Modelo

In [24]:
# O pacote "punkt" é específico para a tarefa de tokenização, que envolve a divisão de um texto
# em uma lista de sentenças
nltk.download("punkt", quiet = True)

True

In [25]:
# Define a métrica
metric = evaluate.load("rouge")

In [26]:
# Função de cálculo das métricas
def ft_calcula_metricas(eval_preds):

    # Desempacota as predições e os rótulos do argumento eval_preds
    previsoes, labels = eval_preds

    # Substitui todos os valores diferentes de -100 em labels pelo ID do token de preenchimento
    labels = np.where(labels != -100,
                      labels,
                      tokenizador.pad_token_id)

    # Decodifica as previsões para texto, ignorando tokens especiais
    previsoes_decodificadas = tokenizador.batch_decode(previsoes,
                                                       skip_special_tokens = True)

    # Decodifica os labels para texto, ignorando tokens especiais
    labels_decodificados = tokenizador.batch_decode(labels,
                                                    skip_special_tokens = True)

    # Adiciona uma nova linha após cada sentença para as previsões decodificadas,
    # preparando-as para a avaliação ROUGE
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in previsoes_decodificadas]

    # Adiciona uma nova linha após cada label para as previsões decodificadas,
    # preparando-os para a avaliação ROUGE
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in labels_decodificados]

    # Calcula a métrica ROUGE entre as previções e os rótulos decodificados, utilizando um stemmer
    resultado = metric.compute(predictions = previsoes_decodificadas,
                               references = labels_decodificados,
                               use_stemmer = True)

    # Retorna o resultado da métrica ROUGE
    return resultado

In [27]:
# Define argumentos de treino
ft_training_args = Seq2SeqTrainingArguments(output_dir = "resultados_treino",
                                             evaluation_strategy = "epoch",
                                             learning_rate = 3e-4,
                                             logging_dir = "logs_treino",
                                             logging_steps = 1,
                                             per_device_train_batch_size = 4,
                                             per_device_eval_batch_size = 2,
                                             weight_decay = 0.01,
                                             save_total_limit = 3,
                                             num_train_epochs = 3,
                                             predict_with_generate = True,
                                             push_to_hub = False)

In [28]:
# Define o trainer
ft_trainer = Seq2SeqTrainer(model = modelo,
                             args = ft_training_args,
                             train_dataset = ft_dataset_tokenized["train"],
                             eval_dataset = ft_dataset_tokenized["test"],
                             tokenizer = tokenizador,
                             data_collator = data_collator,
                             compute_metrics = ft_calcula_metricas)

## Treinamento do Modelo

In [31]:
%%time
ft_trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.611200,2.417329,0.123189,0.028801,0.099276,0.099216
2,2.319200,2.372127,0.121795,0.028101,0.098993,0.098917
3,1.985900,2.366489,0.122118,0.029659,0.098923,0.098813


CPU times: user 28min 25s, sys: 6min 9s, total: 34min 34s
Wall time: 27min 30s


TrainOutput(global_step=1125, training_loss=2.516826660897997, metrics={'train_runtime': 1650.2406, 'train_samples_per_second': 5.441, 'train_steps_per_second': 0.682, 'total_flos': 1525789171924992.0, 'train_loss': 2.516826660897997, 'epoch': 3.0})

In [32]:
# Salva o modelo
ft_trainer.save_model('modelo_salvo')

**ROUGE-1** mede a sobreposição de unigramas (palavras individuais).

**ROUGE-2** mede a sobreposição de bigramas (pares de palavras consecutivas).

**ROUGE-L** mede a sobreposição da subsequência mais longa comum entre o resumo gerado e o resumo de referência. Isso leva em conta a ordem das palavras, mas permite lacunas. Valores mais altos indicam melhor desempenho. ROUGE-L é calculado com base na similaridade entre as sequências, levando em consideração a precisão, o recall e a média harmônica entre eles.

Valores mais altos de ROUGE indicam uma maior semelhança entre o resumo gerado e o resumo de referência, o que geralmente é interpretado como uma indicação de melhor qualidade do resumo. No entanto, é importante lembrar que nenhuma métrica pode capturar completamente a qualidade de um resumo e é útil complementar a avaliação com análises qualitativas ou outras métricas.

## Deploy e Uso do Modelo

In [33]:
# Carrega o tokenizador final salvo em disco
tokenizador_ft_final = AutoTokenizer.from_pretrained('modelo_salvo')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [34]:
# Carrega o modelo final salvo em disco
modelo_ft_final = AutoModelForSeq2SeqLM.from_pretrained('modelo_salvo')

In [35]:
# Visualiza a arquitetura do modelo
modelo_ft_final

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [36]:
# Texto de entrada
texto_entrada = "Can I move out of state with my children if I have a custody agreement in that state?"

In [37]:
# Tokeniza o input
texto_entrada_tokenizado = tokenizador_ft_final(texto_entrada, return_tensors = "pt").input_ids

In [38]:
# Gera a saída (previsão do modelo)
texto_saida_tokenizado = modelo_ft_final.generate(texto_entrada_tokenizado,
                                                   max_length = 50,
                                                   temperature = 0.4,
                                                   do_sample = True)

In [39]:
# Visualiza
texto_saida_tokenizado

tensor([[    0,    37, 16701,  2791,    16,    39,   538,    56,    36, 11293,
             5,   156,    25,    33,  1459,    12,   888,    91,    13,   538,
             6,    25,    56,   174,    12,  1042,     3,     9,  4644,    21,
             3,     9,  7759,     5,   156,    25,    33,     3,  6319,    12,
           129,     3,     9,  7759,     6,    25,   164,    43,  9808,    12]])

In [40]:
# Decode da saída
texto_saida = tokenizador_ft_final.decode(texto_saida_tokenizado[0],
                                           skip_special_tokens = True)

In [41]:
# Resultado
print("Pergunta:", texto_entrada)
print("Resposta:", texto_saida)

Pergunta: Can I move out of state with my children if I have a custody agreement in that state?
Resposta: The custody agreement in your state will be binding. If you are planning to move out of state, you will need to file a motion for a divorce. If you are unable to get a divorce, you may have grounds to


# Fim